In [2]:
from pathlib import Path
import polars as pl

FOLDER_PROYEK = Path("/home/jovyan/work")

DATA_PATH = (
    FOLDER_PROYEK
    / "data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/EmoTweetID-Corpus.csv"
)

FOLDER_PROCESSED = FOLDER_PROYEK / "data/processed"
FOLDER_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Dataset ditemukan:", DATA_PATH.is_file())
print("Folder output:", FOLDER_PROCESSED.resolve())

Dataset ditemukan: True
Folder output: /home/jovyan/work/data/processed


In [3]:
df = pl.scan_csv(str(DATA_PATH))

print("Kolom dataset:", df.collect_schema().names())
df.head(5).collect()

Kolom dataset: ['', 'tweet']


,tweet
i64,str
0,"""Eh l nas l watya ely f hayaty …"
1,"""Lihat, kehidupanku jauh lebih …"
2,"""ramai nya dm"""
3,"""Udane kyk aku mengagumi mu, de…"
4,"""FPI Dibubarkan, Kembali Viral …"


In [4]:
df_clean = (
    df.with_columns(
        pl.col("tweet")
        .cast(pl.String)
        .str.strip_chars()
        .alias("tweet")
    )
    .filter(
        pl.col("tweet").is_not_null()
        & (pl.col("tweet") != "")
    )
    .unique(subset=["tweet"], keep="first")
)

print("Contoh data setelah cleaning:")
df_clean.head(5).collect()

Contoh data setelah cleaning:


,tweet
i64,str
46375,"""Bayangin yang selama ini kamu …"
46515,"""Ni intrusa"""
46328,"""Ryan Raburn."""
46237,"""Page 366 of 366 Hai, terimakas…"
46402,"""Sir Lewis Hamilton, apenas"""


In [5]:
OUTPUT_PATH = FOLDER_PROCESSED / "EmoTweetID_clean.csv"

df_clean.sink_csv(str(OUTPUT_PATH))

print("Data bersih disimpan di:", OUTPUT_PATH.resolve())
print(f"Ukuran file: {OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB")

Data bersih disimpan di: /home/jovyan/work/data/processed/EmoTweetID_clean.csv
Ukuran file: 208.05 MB
